# CSV文件读取 `CsvHelper`

## 成员
- `std::ifstream _ifs`：文件输入流，用于读取CSV文件内容
- `char _buffer[1024]`：行数据缓冲区，存储当前读取的行内容
- `std::string _item_splitter`：字段分隔符字符串
- `std::unordered_map<std::string, int32_t> _fields_map`：字段名到列索引的映射表，用于按字段名查找数据
- `std::vector<std::string> _current_cells`：当前行的字段值列表，存储解析后的各个字段值

## 方法
- **核心属性与构造函数**
  - 构造函数: `CsvReader(const char* item_splitter = ",")`
  - 获取字段（列）总数: `inline uint32_t col_count()`
  - 获取所有字段名称（逗号分隔）: `const char* fields() const`
- **文件加载与迭代**
  - 从文件加载并解析表头: `bool load_from_file(const char* filename)`
  - 读取并解析下一行数据: `bool next_row()`
- **按列索引获取数据**
  - 获取32位有符号整数: `int32_t get_int32(int32_t col)`
  - 获取32位无符号整数: `uint32_t get_uint32(int32_t col)`
  - 获取64位有符号整数: `int64_t get_int64(int32_t col)`
  - 获取64位无符号整数: `uint64_t get_uint64(int32_t col)`
  - 获取双精度浮点数: `double get_double(int32_t col)`
  - 获取字符串: `const char* get_string(int32_t col)`
- **按字段名获取数据**
  - 获取32位有符号整数: `int32_t get_int32(const char* field)`
  - 获取32位无符号整数: `uint32_t get_uint32(const char* field)`
  - 获取64位有符号整数: `int64_t get_int64(const char* field)`
  - 获取64位无符号整数: `uint64_t get_uint64(const char* field)`
  - 获取双精度浮点数: `double get_double(const char* field)`
  - 获取字符串: `const char* get_string(const char* field)`

# 日志管理器 `WTSLogger`
可以通过配置文件构造多个日志器
- 层次
  - 静态日志器 `root`：默认输出目标
  - 其余静态日志器
  - `dyn_pattern`
    - 动态构建日志器模板
- 配置
  - 静态日志器
    - `async`：日志消息是否立即处理，如果不是则被放入一个队列，由一个专门的后台线程负责（降低对主线程的影响）
    - `level`：该日志器只处理 `level` 及以上级别的日志
    - `sinks`：输出配置，可包含多个配置，对于每个配置
      - `filename`：日志文件路径
      - `pattern`：日志格式
      - `level`
      - `type`：输出种类
        - `basic_file_sink`：输出到一个文件中
        - `daily_file_sink`：每天输出到一个新的文件，例如 `filename_2024-09-18.log`
        - `console_sink`：输出到控制台
        - `ostream_sink`：可以输出到任何流对象，这里实现为输出到 std::cout（也是控制台）
  - 动态构建日志器模板
    - `async`
    - `level`
    - `sinks`
      - `filename`：这里含有占位符，用于实际动态构建时替换
      - `pattern`
      - `level`
      - `type`

## 配置文件例子
```JSON
{
    // root 日志器是必须的，它是所有日志的根节点和默认输出目标。
    // 这里配置了一个复杂的 root 日志器，演示了多目标输出 (Multi-Sinks)。
    "root": {
        "async": true,      // 推荐：对根日志器使用异步模式，以最小化对主线程性能的影响。
        "level": "info",    // 日志器级别 (第一道关卡): root 日志器只处理 info 及以上级别的日志。
        "sinks": [
            {
                // 第一个输出目标：每日滚动的日志文件，记录所有通过第一道关卡的日志。
                "type": "daily_file_sink",
                "filename": "logs/wondertrader.log", // 日志文件名，每天会自动生成如 wondertrader_2024-09-18.log 的文件
                "pattern": "[%Y-%m-%d %H:%M:%S.%e] [%l] [%n] %v" // 日志格式: [时间戳] [级别] [日志器名] 日志内容
            },
            {
                // 第二个输出目标：彩色控制台输出，但有更严格的级别过滤。
                "type": "console_sink",
                "level": "warn", // Sink 级别 (第二道关卡): 只有 warn 及以上级别的日志才会显示在控制台。
                "pattern": "[%H:%M:%S.%e] [%^%l%$] %v" // %^...%$ 用于给日志级别着色
            },
            {
                // 第三个输出目标：演示 ostream_sink，同样输出到控制台。
                // 它与 console_sink 类似，但更通用，可以输出到任何 ostream。
                "type": "ostream_sink",
                "level": "fatal", // Sink 级别 (第二道关卡): 只有最严重的 fatal 级别日志才会通过这个 sink 输出。
                "pattern": "!!! FATAL !!! [%Y-%m-%d %H:%M:%S.%e] %v"
            }
        ]
    },

    // 为交易模块定义的静态分类日志器。
    "trading": {
        "async": true,
        "level": "debug", // 设置为 debug 级别，以捕获最详细的交易执行信息供复盘使用。
        "sinks": [
            {
                "type": "daily_file_sink",
                "filename": "logs/trading.log",
                "pattern": "[%Y-%m-%d %H:%M:%S.%e] [%l] %v"
            }
        ]
    },

    // 为风控模块定义的静态分类日志器。
    "risk_control": {
        "async": false, // 设置为同步模式，确保风控相关的日志（特别是错误和警告）被立即写入磁盘，防止程序崩溃时信息丢失。
        "level": "info",
        "sinks": [
            {
                "type": "basic_file_sink",
                "filename": "logs/risk.log",
                "truncate": false, // 不清空文件，持续追加日志。
                "pattern": "[%Y-%m-%d %H:%M:%S.%e] [%l] %v"
            }
        ]
    },

    // 为数据下载/行情接收模块定义的日志器。
    "data_retriever": {
        "async": true,
        "level": "info",
        "sinks": [
            {
                "type": "basic_file_sink",
                "filename": "logs/data.log",
                "truncate": true, // 每次程序启动时清空日志文件，适用于调试数据连接问题，只关心当次运行的日志。
                "pattern": "[%Y-%m-%d %H:%M:%S.%e] %v"
            }
        ]
    },

    // dyn_pattern 是一个特殊的顶级键，用于定义动态日志的“模板”。
    // 这里定义的不是具体的日志器，而是供程序在运行时按需创建日志器的蓝图。
    "dyn_pattern": {
        // 模板1：用于为每个策略实例创建独立的日志文件。
        "strategy_template": {
            "async": true,
            "level": "debug",
            "sinks": [
                {
                    "type": "daily_file_sink",
                    // 核心: filename 中的 "%s" 是一个占位符。
                    // 在运行时，它会被 log_dyn 函数传入的具体分类名 (如 "MyStrategy_rb2410") 替换。
                    // 这样每个策略实例就会有自己的日志文件，例如：logs/strategies/MyStrategy_rb2410.log
                    "filename": "logs/strategies/%s.log",
                    "pattern": "[%Y-%m-%d %H:%M:%S.%e] [%l] %v"
                }
            ]
        },

        // 模板2：用于追踪单个订单的生命周期，方便精细化调试。
        "order_trace_template": {
            "async": false, // 同步写入，确保订单状态的实时记录。
            "level": "debug",
            "sinks": [
                {
                    "type": "basic_file_sink",
                    // %s 会被替换为订单的本地ID (localid)。
                    // 例如：logs/orders/10001.log
                    "filename": "logs/orders/%s.log",
                    "truncate": true, // 每次追踪（可以理解为程序重启后）都生成一个全新的日志文件。
                    "pattern": "[%H:%M:%S.%e] %v"
                }
            ]
        }
    }
}
```

## 成员以及日志管理器存储位置
- **状态与控制标志**
    - `static bool m_bInited;`：初始化标志位，标记日志系统是否已经通过 init() 函数成功初始化
    - `static bool m_bTpInited;`：异步线程池初始化标志位
      - 当配置文件中任何一个日志器被配置为异步模式时，就需要一个后台线程池来处理日志写入队列
    - `static bool m_bStopped;`：日志系统停止标志位，当被设为 `true` 时，所有日志输出调用都将被忽略
- **核心处理组件**
    - `static ILogHandler* m_logHandler;`：自定义日志处理器指针
      - 可以通过 `init()` 传入，或者 `registerHandler()` 传入一个实现了 `ILogHandler` 接口的对象
      - 之后每一条日志被传入日志处理器，也会被传递给该处理器
    - `static WTSLogLevel m_logLevel;`：全局日志级别过滤器，作为日志的第一道关卡，低于此级别的日志消息会被直接丢弃
    - `static SpdLoggerPtr m_rootLogger;`：根日志器 root 的智能指针，作为所有未指定分类日志的默认输出目标
    - `static thread_local char m_buffer[MAX_LOG_BUF_SIZE];`：线程本地日志缓冲区，每个线程独享一个2KB的缓冲区用于格式化日志消息
- **动态日志管理**
    - `static LogPatterns* m_mapPatterns;`：动态日志模板映射表
      - `LogPatterns` 是 `WTSHashMap<std::string>` 的别名
      - 以模板名称为键，存储从配置文件 `dyn_pattern` 中解析出的 `WTSVariant` 配置对象
    - `static std::set<std::string> m_setDynLoggers;`：动态日志器名称集合
      - 用于追踪所有通过模板在运行时动态创建的日志器的名称

`WTSLogger` 并不直接持有除 `m_rootLogger`（根日志器 root）外的所有日志器实例
- 无论是从配置文件中解析出的静态日志器，还是运行时动态创建的日志器
- 都会通过 `spdlog::register_logger()` 函数注册并存储在 `spdlog` 库内部的一个 *全局静态的哈希表（注册表）* 中
- 当需要使用某个日志器时，`WTSLogger` 会通过 `spdlog::get(logger_name)` 按名称从这个全局注册表中获取

## 方法
- **日志系统管理接口**
    - ***初始化日志系统***
        ```cpp
        /* @param propFile 配置文件路径或配置内容字符串，默认为"logcfg.json"
        * @param isFile 指示propFile参数的类型，true表示文件路径，false表示配置内容
        * @param handler 可选的自定义日志处理器，用于接收所有日志消息
        */
        void WTSLogger::init(const char* propFile /* = "logcfg.json" */, bool isFile /* = true */, ILogHandler* handler /* = NULL */)
        ```
    - ***注册自定义日志处理器***：`static void registerHandler(ILogHandler* handler = NULL) {m_logHandler = handler;}`
    - ***停止日志系统***：`static void stop()`
    - ***释放所有动态创建的日志器***：`static void freeAllDynLoggers()`
- **原始日志输出接口**
    - ***输出原始日志消息到根日志器 root、自定义日志处理器 m_logHandler***
        ```cpp
        /*@param ll 日志级别，用于级别过滤和选择输出方法
        * @param message 已格式化的日志消息内容，不再进行格式化处理
        */
        void WTSLogger::log_raw(WTSLogLevel ll, const char* message)
        ```
    - ***输出原始日志消息到对应静态日志器、根日志器 root、自定义日志处理器 m_logHandler***
        ```cpp
        /*@param catName 日志器名称，用于获取对应的日志器
        * @param ll 日志级别，用于级别过滤和选择输出方法
        * @param message 已格式化的日志消息内容，不再进行格式化处理
        */
       void WTSLogger::log_raw_by_cat(const char* catName, WTSLogLevel ll, const char* message)
        ```
    - ***输出原始日志消息到对应动态日志器（如果没有则先创建）、根日志器 root、自定义日志处理器 m_logHandler***
        ```cpp
        /* @param patttern 动态日志模式名称，用于查找对应的配置模板
        * @param catName 日志分类名称，将作为动态创建的日志器名称
        * @param ll 日志级别，用于级别过滤和选择输出方法
        * @param message 已格式化的日志消息内容，不再进行格式化处理
        */
        void WTSLogger::log_dyn_raw(const char* patttern, const char* catName, WTSLogLevel ll, const char* message)
        ```
- **格式化日志输出接口**
    - ***输出调试级别的格式化日志到根日志器 root、自定义日志处理器 m_logHandler***
        ```cpp
        /* @tparam Args 可变参数模板类型
        * @param format 格式化字符串（支持fmt格式）
        * @param args 格式化参数
        * 例如：WTSLogger::debug("用户{}登录，时间:{}", userId, timestamp);
        */
        template<typename... Args>
        static void debug(const char* format, const Args& ...args)
        ```
    - ***输出信息级别的格式化日志到根日志器 root、自定义日志处理器 m_logHandler***：`...info...`
    - ***输出警告级别的格式化日志到根日志器 root、自定义日志处理器 m_logHandler***：`...warn...`
    - ***输出错误级别的格式化日志到根日志器 root、自定义日志处理器 m_logHandler***：`...error...`
    - ***输出致命错误级别的格式化日志到根日志器 root、自定义日志处理器 m_logHandler***：`...fatal...`
    - ***输出指定级别的格式化日志到根日志器 root、自定义日志处理器 m_logHandler***：`...log(WTSLogLevel ll, const char* format, const Args& ...args)`
    - ***输出指定级别的格式化日志到指定静态日志器、根日志器 root、自定义日志处理器 m_logHandler***
        ```cpp
        /*@tparam Args 可变参数模板类型
        * @param catName 日志分类名称
        * @param ll 日志级别
        * @param format 格式化字符串（支持fmt格式）
        * @param args 格式化参数
        */
        template<typename... Args>
        static void log_by_cat(const char* catName, WTSLogLevel ll, const char* format, const Args& ...args)
        ```
    - ***输出指定级别的格式化日志（且前缀带 "[级别]"）到指定静态日志器、自定义日志处理器 m_logHandler*** `...log_by_cat_prefix...`
    - ***输出指定级别的格式化日志到对应动态日志器（如果没有则先创建）、根日志器 root、自定义日志处理器 m_logHandler***
        ```cpp
        /* @tparam Args 可变参数模板类型
        * @param patttern 日志模式名称
        * @param catName 日志分类名称
        * @param ll 日志级别
        * @param format 格式化字符串（支持fmt格式）
        * @param args 格式化参数
        */
        template<typename... Args>
        static void log_dyn(const char* patttern, const char* catName, WTSLogLevel ll, const char* format, const Args& ...args)
        ```
    - ***输出指定级别的格式化日志（且前缀带 "[级别]"）到对应动态日志器（如果没有则先创建）、根日志器 root、自定义日志处理器 m_logHandler***：`...log_dyn_prefix...`
- **内部实现方法**
    - ***输出调试级别日志到指定日志器、根日志器、自定义日志处理器 m_logHandler***
        ```cpp
        /* @param logger 目标日志器指针，可以为NULL
        * @param message 已格式化的日志消息内容
        */
        void WTSLogger::debug_imp(SpdLoggerPtr logger, const char* message)
        {
            // 如果指定的日志器存在，输出调试信息到该日志器
            if (logger)
                logger->debug(message);

            // 如果指定的日志器不是根日志器，同时输出到根日志器
            if (logger != m_rootLogger)
                m_rootLogger->debug(message);

            // 如果存在自定义日志处理器，调用其处理方法
            if (m_logHandler)
                m_logHandler->handleLogAppend(LL_DEBUG, message);
        }
        ```
    - ***输出信息级别日志到指定日志器、根日志器、自定义日志处理器 m_logHandler***：`...info_imp...`
    - ***输出警告级别日志到指定日志器、根日志器、自定义日志处理器 m_logHandler***：`...warn_imp...`
    - ***输出错误级别日志到指定日志器、根日志器、自定义日志处理器 m_logHandler***：`...error_imp...`
    - ***输出致命级别日志到指定日志器、根日志器、自定义日志处理器 m_logHandler***：`...fatal_imp...`
    - ***初始化指定分类的日志器***：`static void initLogger(const char* catName, WTSVariant* cfgLogger)`
    - ***获取指定名称的日志器***：`static SpdLoggerPtr getLogger(const char* logger, const char* pattern = "")`
    - ***在控制台打印消息（未初始化时使用）***：`static void print_message(const char* buffer)`

# 基础数据管理器 `WTSBaseDataMgr`
管理交易系统运行的基础信息，包括：合约信息、品种信息、交易时间、节假日等。

继承了 `IBaseDataMgr`，参考 [Includes/note.ipynb/数据管理接口层/基础数据管理接口 IBaseDataMgr](../Includes/note.ipynb)

## 成员
- `TradingDayTplMap m_mapTradingDay`：交易日模板映射表，存储各地区的交易日历模板
  - typedef wt_hashmap\<std::string, `TradingDayTpl`\>	TradingDayTplMap;
    - uint32_t _cur_tdate：当前交易日日期，格式为YYYYMMDD
	- `HolidaySet` _holidays：节假日集合，存储所有节假日日期
    	- typedef wt_hashset\<uint32_t\> HolidaySet;
  - 本质上是 **map<地区ID，(当前交易日期，set(节假日ID))>**
- `SessionCodeMap m_mapSessionCode`：时段代码映射表，维护时段与品种的对应关系
  - typedef wt_hashmap\<std::string, `CodeSet`\> SessionCodeMap;
    - typedef fastest_hashset\<std::string\> CodeSet;
  - 本质上是 **map<时段ID，set(品种ID)>**
- `WTSExchgContract* m_mapExchgContract`：按交易所组织的合约信息映射表
  - 本质上是 **map<交易所ID，map\*\<合约ID，合约信息WTSContractInfo\*\>\>**
- `WTSSessionMap* m_mapSessions`：交易时段信息映射表
  - 本质上是 **map<品种ID, 品种交易时段信息WTSSessionInfo\*\>**
- `WTSCommodityMap* m_mapCommodities`：商品品种信息映射表
  - 本质上是 **map<品种ID, 品种信息WTSCommodityInfo\*\>**
- `WTSContractMap* m_mapContracts`：合约信息映射表，支持同名合约的多版本管理
  - 本质上是 **map<合约ID, array(合约信息WTSContractInfo\*\)\>**
  - 因为一个合约可能在多个交易所都有

## 方法

### 核心属性与构造函数

#### 构造函数 
```cpp
WTSBaseDataMgr::WTSBaseDataMgr()
	: m_mapExchgContract(NULL)    // 交易所合约映射表指针初始化为NULL
	, m_mapSessions(NULL)         // 交易时段映射表指针初始化为NULL
	, m_mapCommodities(NULL)      // 商品品种映射表指针初始化为NULL
	, m_mapContracts(NULL)        // 合约映射表指针初始化为NULL
{
	m_mapExchgContract = WTSExchgContract::create();
	m_mapSessions = WTSSessionMap::create();
	m_mapCommodities = WTSCommodityMap::create();
	m_mapContracts = WTSContractMap::create();
}
```

#### 加载交易时段配置 loadSessions
配置文件格式（JSON）示例：
```JSON
{
	"DAY": {
		"name": "日盘",
		"offset": 0,
		"auction": {"from": 85500, "to": 90000},
		"sections": [
				{"from": 90000, "to": 101500},
				{"from": 103000, "to": 150000}
			]
	}
}
```

配置项说明：
- id：时段标识符（如"DAY"、"NIGHT"）
- name：时段显示名称
- offset：时区偏移量（小时）
- auction/auctions：集合竞价时间段
- sections：连续交易时间段数组

```cpp
/**
 * @brief 加载交易时段配置
 * @param filename 交易时段配置文件路径
 * @return bool 加载成功返回true，失败返回false
 * 
 * 该函数从配置文件中加载所有交易时段的定义，是系统初始化的关键步骤之一。
 * 交易时段配置包含了各个品种的交易时间规则，是交易系统正常运行的基础。
 * 

 * 
 * 加载过程：
 * 1. 检查配置文件是否存在
 * 2. 解析JSON格式的配置文件
 * 3. 遍历所有时段定义
 * 4. 创建WTSSessionInfo对象
 * 5. 设置集合竞价时间
 * 6. 添加连续交易时间段
 * 7. 将时段信息加入映射表
 */
bool WTSBaseDataMgr::loadSessions(const char* filename)
{
	// 检查交易时段配置文件是否存在
	if (!StdFile::exists(filename))
	{
		WTSLogger::error("Trading sessions configuration file {} not exists", filename);
		return false;  // 文件不存在，加载失败
	}

	// 使用配置加载器解析配置文件
	// WTSCfgLoader支持JSON、INI等多种格式
	WTSVariant* root = WTSCfgLoader::load_from_file(filename);
	if (root == NULL)
	{
		WTSLogger::error("Loading session config file {} failed", filename);
		return false;  // 配置文件解析失败
	}

	// 遍历配置文件中的所有交易时段定义
	for(const std::string& id : root->memberNames())
	{
		// 获取当前时段的配置对象
		WTSVariant* jVal = root->get(id);

		// 从配置中读取时段的基本信息
		const char* name = jVal->getCString("name");    // 时段显示名称
		int32_t offset = jVal->getInt32("offset");      // 时区偏移量

		// 创建交易时段信息对象
		WTSSessionInfo* sInfo = WTSSessionInfo::create(id.c_str(), name, offset);

		// 处理集合竞价时间配置
		// 支持单个集合竞价时间段（auction）和多个集合竞价时间段（auctions）
		if (jVal->has("auction"))
		{
			// 单个集合竞价时间段配置
			WTSVariant* jAuc = jVal->get("auction");
			sInfo->setAuctionTime(jAuc->getUInt32("from"), jAuc->getUInt32("to"));
		}
		else if (jVal->has("auctions"))
		{
			// 多个集合竞价时间段配置（支持多次集合竞价）
			WTSVariant* jAucs = jVal->get("auctions");
			for (uint32_t i = 0; i < jAucs->size(); i++)
			{
				WTSVariant* jSec = jAucs->get(i);
				sInfo->addAuctionTime(jSec->getUInt32("from"), jSec->getUInt32("to"));
			}
		}

		// 处理连续交易时间段配置
		WTSVariant* jSecs = jVal->get("sections");
		if (jSecs == NULL || !jSecs->isArray())
			continue;  // 没有交易时间段配置，跳过当前时段

		// 遍历所有连续交易时间段
		for (uint32_t i = 0; i < jSecs->size(); i++)
		{
			WTSVariant* jSec = jSecs->get(i);
			// 添加交易时间段（开始时间、结束时间）
			// 时间格式：HHMMSS（如90000表示9:00:00）
			sInfo->addTradingSection(jSec->getUInt32("from"), jSec->getUInt32("to"));
		}

		// 将配置好的交易时段信息添加到映射表中
		m_mapSessions->add(id.c_str(), sInfo);
	}

	// 释放配置文件解析产生的根对象，避免内存泄漏
	root->release();

	// 所有交易时段配置加载成功
	return true;
}
```

#### 加载商品品种配置

#### 加载合约信息配置

#### 加载节假日配置

#### 释放所有资源

### 合约与品种查询接口
    - 按标准ID获取品种信息: `virtual WTSCommodityInfo* getCommodity(const char* stdPID)`
    - 按交易所和品种代码获取品种信息: `virtual WTSCommodityInfo* getCommodity(const char* exchg, const char* pid)`
    - 获取单个合约信息: `virtual WTSContractInfo* getContract(const char* code, const char* exchg = "", uint32_t uDate = 0)`
    - 获取合约列表: `virtual WTSArray* getContracts(const char* exchg = "", uint32_t uDate = 0)`
    - 获取合约数量: `virtual uint32_t getContractSize(const char* exchg = "", uint32_t uDate = 0)`

### 交易时段查询接口
    - 按ID获取交易时段信息: `virtual WTSSessionInfo* getSession(const char* sid)`
    - 按合约代码获取交易时段信息: `virtual WTSSessionInfo* getSessionByCode(const char* code, const char* exchg = "")`
    - 获取所有交易时段信息: `virtual WTSArray* getAllSessions()`
    - 获取使用某时段的品种集合: `CodeSet* getSessionComms(const char* sid)`

### 交易日历与日期计算
    - 判断是否为节假日: `virtual bool isHoliday(const char* stdPID, uint32_t uDate, bool isTpl = false)`
    - 判断是否为交易日: `bool isTradingDate(const char* stdPID, uint32_t uDate, bool isTpl = false)`
    - 根据自然时间计算所属交易日: `virtual uint32_t calcTradingDate(const char* stdPID, uint32_t uDate, uint32_t uTime, bool isSession = false)`
    - 获取当前交易日: `uint32_t getTradingDate(const char* stdPID, uint32_t uOffDate = 0, uint32_t uOffMinute = 0, bool isTpl = false)`
    - 设置当前交易日: `void setTradingDate(const char* stdPID, uint32_t uDate, bool isTpl = false)`
    - 获取下一个交易日: `uint32_t getNextTDate(const char* stdPID, uint32_t uDate, int days = 1, bool isTpl = false)`
    - 获取上一个交易日: `uint32_t getPrevTDate(const char* stdPID, uint32_t uDate, int days = 1, bool isTpl = false)`
    - 获取交易日边界时间（开/收盘）: `virtual uint64_t getBoundaryTime(const char* stdPID, uint32_t tDate, bool isSession = false, bool isStart = true)`